In [12]:
from src.models.lm import DecoderLM

model_params = {
    "vocab_size": 48000,         # replace with actual vocab size if needed
    "seq_len": 1024,             # replace with actual seq_len if needed
    "d_model": 1792,
    "d_ff": 4928,
    "n_heads": 28,
    "use_gqa": True,
    "gqa_factor": 4,
    "N": 26,
    "p_dropout": 0.0,
    "norm_type": "rms",
    "pos_emb_type": "rope",
    "rope_base": 10000,
    "ff_type": "swiglu",
    "use_bias": False,
    "intra_doc_mask": False,
    "compile_model": True,
    "compile_mode": "default",    # "max-autotune-no-cudagraphs"
    "compile_loss": True,
    "use_torch_sdpa": True,
    "fused_ce": False,
}

model = DecoderLM(**model_params)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 983,659,264


In [15]:
import torch

# ~1B architectures, deep/narrow -> shallow/wide
# head_dim = 64 (n_heads = d_model // 64), d_ff = 2.75 * d_model, gqa_factor in {1 (no gqa), 2, 3, 4}
overrides = [
    {"d_model": 1536, "d_ff": 4224, "n_heads": 24, "N": 35, "use_gqa": True, "gqa_factor": 2},
    {"d_model": 1536, "d_ff": 4224, "n_heads": 24, "N": 36, "use_gqa": True, "gqa_factor": 3},
    {"d_model": 1536, "d_ff": 4224, "n_heads": 24, "N": 37, "use_gqa": True, "gqa_factor": 4},
    {"d_model": 1664, "d_ff": 4576, "n_heads": 26, "N": 30, "use_gqa": True, "gqa_factor": 2},
    {"d_model": 1728, "d_ff": 4752, "n_heads": 27, "N": 28, "use_gqa": True, "gqa_factor": 3},
    {"d_model": 1792, "d_ff": 4928, "n_heads": 28, "N": 25, "use_gqa": True, "gqa_factor": 2},
    {"d_model": 1792, "d_ff": 4928, "n_heads": 28, "N": 26, "use_gqa": True, "gqa_factor": 4},
    {"d_model": 1920, "d_ff": 5280, "n_heads": 30, "N": 22, "use_gqa": True, "gqa_factor": 2},
    {"d_model": 2048, "d_ff": 5632, "n_heads": 32, "N": 19, "use_gqa": True, "gqa_factor": 2},
    {"d_model": 2048, "d_ff": 5632, "n_heads": 32, "N": 20, "use_gqa": True, "gqa_factor": 4},
    # no gqa
    {"d_model": 1536, "d_ff": 4224, "n_heads": 24, "N": 32, "use_gqa": False, "gqa_factor": 1},
    {"d_model": 1664, "d_ff": 4576, "n_heads": 26, "N": 27, "use_gqa": False, "gqa_factor": 1},
    {"d_model": 1728, "d_ff": 4752, "n_heads": 27, "N": 25, "use_gqa": False, "gqa_factor": 1},
    {"d_model": 1792, "d_ff": 4928, "n_heads": 28, "N": 23, "use_gqa": False, "gqa_factor": 1},
    {"d_model": 1920, "d_ff": 5280, "n_heads": 30, "N": 20, "use_gqa": False, "gqa_factor": 1},
]

for o in overrides:
    assert o["n_heads"] * 64 == o["d_model"] and o["d_ff"] == 2.75 * o["d_model"] and o["n_heads"] % o["gqa_factor"] == 0
    with torch.device("meta"):
        m = DecoderLM(**{**model_params, **o})
    n = sum(p.numel() for p in m.parameters())
    print(f"{n:>14,}  {(n - 1e9) / 1e7:+.1f}%   N_{o['N']}, dim_{o['d_model']}, mlp_{o['d_ff']}, gqa{o['gqa_factor']}")
    del m

 1,002,809,856  +0.3%   N_35, dim_1536, mlp_4224, gqa2
 1,001,043,456  +0.1%   N_36, dim_1536, mlp_4224, gqa3
 1,012,253,184  +1.2%   N_37, dim_1536, mlp_4224, gqa4
 1,014,475,904  +1.4%   N_30, dim_1664, mlp_4576, gqa2
   995,758,272  -0.4%   N_28, dim_1728, mlp_4752, gqa3
   989,275,392  -1.1%   N_25, dim_1792, mlp_4928, gqa2
   983,659,264  -1.6%   N_26, dim_1792, mlp_4928, gqa4
 1,004,630,400  +0.5%   N_22, dim_1920, mlp_5280, gqa2
   994,916,352  -0.5%   N_19, dim_2048, mlp_5632, gqa2
 1,000,163,328  +0.0%   N_20, dim_2048, mlp_5632, gqa4
   998,671,872  -0.1%   N_32, dim_1536, mlp_4224, gqa1
   995,775,872  -0.4%   N_27, dim_1664, mlp_4576, gqa1
   997,489,728  -0.3%   N_25, dim_1728, mlp_4752, gqa1
   990,873,856  -0.9%   N_23, dim_1792, mlp_4928, gqa1
   995,406,720  -0.5%   N_20, dim_1920, mlp_5280, gqa1
